In [17]:
# import modules
import numpy as np
import tensorflow as tf
from tensorflow import keras
import pandas as pd
import seaborn as sns
from pylab import rcParams
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from scipy import stats
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras import optimizers
from datetime import datetime
from kerastuner.tuners import RandomSearch
import tensorflow_addons as tfa
from keras.utils import to_categorical 
from keras.regularizers import l2




r_seed = 42

WORKDIR = 'C:/Users/thewr/git/LSTM_giroscopio_acelerometro.git/'
np.random.seed(r_seed)
tf.random.set_seed(r_seed)
data_hora= datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
LOG_DIR = WORKDIR + "/Code/DataPrep/"+data_hora

In [18]:

# Assuming your data is saved as 'data.csv'
# Load the data from the CSV file using pandas
df = pd.read_csv('df_X_train_1615_1650_70_percent.csv')



In [19]:
df.head()

,1,2,3,4,5,6
0,8.906280,-5.800103,-0.044293,0.418050,-1.053929,1.149950
1,7.931840,-1.045219,-2.230199,-1.676260,-0.494665,1.602687
2,6.737134,-10.686669,1.425747,1.942443,-0.653389,-0.230633
3,2.834585,1.487847,-2.479196,1.781588,-1.616388,-1.506820
4,7.105841,-1.389984,1.528698,0.279565,-0.964446,-1.309746


In [20]:
df.shape

(22035800, 6)

In [21]:
# Extract the actual 3D array from the DataFrame


X_train = df.to_numpy().reshape(110179, 200, 6)
#data = df.to_numpy().reshape(2,3,4)

print("3D array loaded and reshaped back to its original dimensions.")
print(X_train.shape)  # Output: (2, 3, 4)

3D array loaded and reshaped back to its original dimensions.
(110179, 200, 6)


In [22]:
# Assuming your data is saved as 'data.csv'
# Load the data from the CSV file using pandas
df_xval = pd.read_csv('df_X_validation_1600_1614_30_percent.csv')
print(df_xval.shape)

# Extract the actual 3D array from the DataFrame


X_validation = df_xval.to_numpy().reshape(46822, 200, 6)
#data = df.to_numpy().reshape(2,3,4)

print("3D array loaded and reshaped back to its original dimensions.")
print(X_validation.shape)  # Output: (2, 3, 4)

(9364400, 6)
3D array loaded and reshaped back to its original dimensions.
(46822, 200, 6)


In [23]:
# Assuming your data is saved as 'data.csv'
# Load the data from the CSV file using pandas
df_ytrain = pd.read_csv('df_y_train__1615_1650_70_percent.csv')
print(df_ytrain.shape)

# Extract the actual 3D array from the DataFrame


y_train = df_ytrain.to_numpy().reshape(110179,18)
#data = df.to_numpy().reshape(2,3,4)

print("3D array loaded and reshaped back to its original dimensions.")
print(y_train.shape)  # Output: (2, 3, 4)

(110179, 18)
3D array loaded and reshaped back to its original dimensions.
(110179, 18)


In [24]:
# Assuming your data is saved as 'data.csv'
# Load the data from the CSV file using pandas
df_yval = pd.read_csv('df_y_validation_1600_1614_30_percent.csv')

# Extract the actual 3D array from the DataFrame


y_validation = df_yval.to_numpy().reshape(df_yval.shape[0], 18)
#data = df.to_numpy().reshape(2,3,4)

print("3D array loaded and reshaped back to its original dimensions.")
print(y_validation.shape)  # Output: (2, 3, 4)

3D array loaded and reshaped back to its original dimensions.
(46822, 18)


In [25]:
print(X_train.shape)
print(y_train.shape)
print(X_validation.shape)
print(y_validation.shape)

(110179, 200, 6)
(110179, 18)
(46822, 200, 6)
(46822, 18)


In [26]:
def shuffle_data(variable_data,label_data):
    perm_indices = np.random.permutation(variable_data.shape[0])
    # Shuffle both arrays using the same permutation indices
    shuffled_variable_data = variable_data[perm_indices]
    shuffled_label_data = label_data[perm_indices]
    return (shuffled_variable_data,shuffled_label_data)

In [27]:
#X_train,y_train = shuffle_data(X_train,y_train)
#X_validation,y_validation = shuffle_data(X_validation,y_validation)

In [28]:
""" print(X_train.shape)
print(y_train.shape)
print(X_validation.shape)
print(y_validation.shape) """

' print(X_train.shape)\nprint(y_train.shape)\nprint(X_validation.shape)\nprint(y_validation.shape) '

In [29]:
#from tensorflow.keras import regularizers

In [30]:
def cria_LSTM():
  model = tf.keras.Sequential()  
  model.add(tf.keras.layers.LSTM(units=224, kernel_regularizer=l2(0.0002729031103694528),
             recurrent_regularizer=l2(0.0002729031103694528), input_shape=(200,6), return_sequences=False))  
  model.add(tf.keras.layers.Dense(units = 96, activation='relu'))  
  model.add(tf.keras.layers.Dense(units=18, activation='softmax'))
  model.add(tf.keras.layers.Flatten())
  model.compile(loss='categorical_crossentropy', optimizer=tf.keras.optimizers.Adam(learning_rate=0.0010277342442932575), metrics=['accuracy'])
  return model



In [31]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=7,
    min_delta = 0,
    verbose=1,
    restore_best_weights=True)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=1,
    verbose=1)

tensorboard = tf.keras.callbacks.TensorBoard(LOG_DIR)

my_callbacks = [early_stopping]  


In [32]:
model  = cria_LSTM()
model.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_validation, y_validation),use_multiprocessing=True,callbacks=[my_callbacks])  # Adjust epochs and batch size
loss, accuracy = model.evaluate(X_validation, y_validation)

Epoch 1/50
3444/3444 [==============================] - 1084s 314ms/step - loss: 1.2908 - accuracy: 0.6046 - val_loss: 1.3990 - val_accuracy: 0.6512
Epoch 2/50
3444/3444 [==============================] - 1168s 339ms/step - loss: 0.7936 - accuracy: 0.7807 - val_loss: 1.4274 - val_accuracy: 0.6739
Epoch 3/50
3444/3444 [==============================] - 1152s 334ms/step - loss: 0.6365 - accuracy: 0.8402 - val_loss: 1.6322 - val_accuracy: 0.6720
Epoch 4/50
3444/3444 [==============================] - 1172s 340ms/step - loss: 0.5522 - accuracy: 0.8723 - val_loss: 1.8173 - val_accuracy: 0.6422
Epoch 5/50
  35/3444 [..............................] - ETA: 16:12 - loss: 0.5941 - accuracy: 0.8598

KeyboardInterrupt: 

In [ ]:
model.results_summary()

In [ ]:
loss, accuracy = model.evaluate(X_validation, y_validation)

In [ ]:
predictions = model.predict(X_validation)

In [ ]:
print(y_validation)

In [ ]:
print(y_validation.shape)

In [ ]:
y_validation_inverse = np.argmax(y_validation, axis=1)

In [ ]:
print(y_validation_inverse)

In [ ]:
print(len(y_validation_inverse))

In [ ]:
print(predictions)

In [ ]:
classes_x=np.argmax(predictions,axis=1)

In [ ]:
print(classes_x)

In [ ]:
print(len(classes_x))

In [ ]:
print(classes_x.shape)

In [ ]:
from sklearn.metrics import confusion_matrix,ConfusionMatrixDisplay,classification_report
cm = confusion_matrix(y_validation_inverse, classes_x)

In [ ]:
disp = ConfusionMatrixDisplay(confusion_matrix=cm)

In [ ]:
disp.plot()

In [ ]:
print(classification_report(y_validation_inverse, classes_x))

In [ ]:
best_model.save(WORKDIR+"/Model/best_models_saved")

In [ ]:
""" import pandas as pd

# Create a DataFrame, optionally adding column names
df_X_train = pd.DataFrame(X_train.reshape(-1, X_train.shape[-1]))
df_X_train.columns = ["1", "2", "3", "4","5","6","7","8","9","10","11","12","13","14","15","16","17","18","19",
                           "20","21","22","23","24","25","26","27"]  # Optional: Add column names

# Save the DataFrame to a CSV file
df_X_train.to_csv('df_X_train.csv', index=False)

print("3D array saved to df_X_train.csv") """

In [ ]:
""" import pandas as pd

# Create a DataFrame, optionally adding column names
df_X_validation = pd.DataFrame(X_validation.reshape(-1, X_validation.shape[-1]))
df_X_validation.columns = ["1", "2", "3", "4","5","6","7","8","9","10","11","12","13","14","15","16","17","18","19",
                           "20","21","22","23","24","25","26","27"]  # Optional: Add column names

# Save the DataFrame to a CSV file
df_X_validation.to_csv('df_X_validation.csv', index=False)

print("3D array saved to data.csv") """

In [ ]:
""" import pandas as pd

# Create a DataFrame, optionally adding column names
df_X_train = pd.DataFrame(X_train.reshape(-1, X_train.shape[-1]))
df_X_train.columns = ["1", "2", "3", "4","5","6","7","8","9","10","11","12","13","14","15","16","17","18","19",
                           "20","21","22","23","24","25","26","27"]  # Optional: Add column names

# Save the DataFrame to a CSV file
df_X_train.to_csv('df_X_train.csv', index=False)

print("3D array saved to data.csv") """

In [ ]:
""" import pandas as pd

# Create a DataFrame, optionally adding column names
df_y_train = pd.DataFrame(y_train.reshape(-1, y_train.shape[-1]))
df_y_train.columns = ["1", "2", "3", "4","5","6","7","8","9","10","11","12","13","14","15","16","17","18"]  # Optional: Add column names

# Save the DataFrame to a CSV file
df_y_train.to_csv('df_y_train.csv', index=False)

print("3D array saved to df_y_train.csv") """

In [ ]:
""" import pandas as pd

# Create a DataFrame, optionally adding column names
df_y_validation = pd.DataFrame(y_validation.reshape(-1, y_validation.shape[-1]))
df_y_validation.columns = ["1", "2", "3", "4","5","6","7","8","9","10","11","12","13","14","15","16","17","18"]  # Optional: Add column names

# Save the DataFrame to a CSV file
df_y_validation.to_csv('df_y_validation.csv', index=False)

print("3D array saved to y_validation.csv") """